# 환경설정

In [ ]:
# 1. 나눔 폰트 설치
!sudo apt-get install -qq gengetopt fonts-nanum

# 2. 폰트 캐시 정리 (에러 방지)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


In [ ]:
!pip install "paramiko<3.0" sshtunnel psycopg2-binary --upgrade-strategy eager --quiet

In [ ]:
# 2. 라이브러리 임포트 (중복 제거 및 그룹화)
import os
import pandas as pd
import numpy as np
import psycopg2
import gspread
import matplotlib.pyplot as plt
from matplotlib import font_manager
from datetime import datetime, timedelta

from sshtunnel import SSHTunnelForwarder
from google.colab import drive, auth
from google.cloud import bigquery
from google.auth import default

# 3. 구글 인증 및 클라이언트 초기화 (한 번만)
auth.authenticate_user()
drive.mount('/content/drive')

creds, _ = default()
gc = gspread.authorize(creds)
client = bigquery.Client(project='fivespot-bigquery')

#인증 정보

In [ ]:
# 1. SSH Tunnel 및 데이터베이스 접속 설정 (포트폴리오용 마스킹 버전)
ssh_host = "YOUR_SSH_HOST_IP"
ssh_username = "YOUR_SSH_USERNAME"
pem_path = "YOUR_SECURE_STORAGE_PATH/aws-eb.pem"

db_host = "YOUR_AWS_RDS_ENDPOINT.amazonaws.com"
db_port = 5432
db_user = "YOUR_DB_READONLY_USER"
db_password = "YOUR_DB_PASSWORD"
db_name = "YOUR_DATABASE_NAME"

# 데이터 전처리 (재구매 분석에 사용 가능 데이터 추출)

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

        # 4. 쿼리 실행
    df = pd.read_sql("""
                         SELECT
                                    TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS r_date,
                                    TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS r_time,
                                    c.contract_uid,
                                    c.client_uid,
                                    c.actual_price,
                                    ph.price,
                                    c.status,
                                    ph.payment_status,
                                    c.product_name,
                                    pp.name product_period,
                                    p.paid_from,
                                    cl.phone_number,
                                    TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                                    TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                                    TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                                    c.is_migrated,
                                    c.is_deleted,
                                    c.initial_count,
                                    c.remain_count
                              FROM contract c
                              LEFT JOIN price_policy pp
                              ON c.price_policy_uid = pp.price_policy_uid
                              LEFT JOIN payment_history ph
                              ON c.contract_uid = ph.payment_uid
                              LEFT JOIN payment p
                              ON c.contract_uid = p.payment_uid
                              LEFT JOIN client cl
                              ON c.client_uid = cl.client_uid
                              WHERE cl.phone_number NOT IN (SELECT phone_number FROM admin)
                              and c.actual_price != 0
                              and ph.payment_status = 'DONE'
                              ORDER BY ph.requested_at
                    """, conn)
    conn.close()
df.head()

In [ ]:
import requests
colab_ip = requests.get('https://api.ipify.org').text
print(f"현재 코랩 세션의 IP 주소: {colab_ip}")

In [ ]:
df['product_name'] = df['product_name'] + ' - ' + df['product_period'].astype(str)
df = df.drop(columns=['product_period'])
df.head()

In [ ]:
df = df[~df['status'].isin(['WITHDRAW', 'CANCELED'])].copy()
df = df[~df['start_date'].isna()].copy()
df['status'].unique()

In [ ]:
import pandas as pd

def categorize_product(name):
    # 1. 매일 3시간 패스 (최우선 분류)
    if '매일 3시간' in name:
        return '매일 3시간패스'

    # 2. 주말&야간 / 위켄드 패스
    elif '주말&야간' in name or '위켄드' in name:
        return '주말&야간패스'

    # 3. 시간/1회권 (체험형)
    # '회'가 포함되더라도 '1회'인 경우와 '1일'인 경우를 먼저 체크
    elif ('1회' in name) or ('1일' in name) or (name.endswith('패스') and any(h in name for h in ['2시간', '4시간', '6시간', '8시간', '12시간', '24시간']) and '회' not in name):
        # 단, '10회' 등과 겹치지 않게 '1회'를 정확히 체크
        if '10회' in name or '50회' in name or '100회' in name:
            return '차감권(N회)'
        return '시간/1회권'

    # 4. 차감권 (N회 이용형)
    elif '회' in name:
        return '차감권(N회)'

    # 5. 무제한 기간권 (정착형)
    elif '무제한' in name or '개월' in name or '주' in name or '일' in name:
        return '무제한 기간권'

    else:
        return '기타'

# 데이터프레임에 적용
df['상품카테고리'] = df['product_name'].apply(categorize_product)

df.head()

In [ ]:
df['상품카테고리'].unique()

In [ ]:
import pandas as pd

# 1. 우선 전체 데이터 정렬 (유저, 시작일, 결제시간 순)
df = df.sort_values(by=['client_uid', 'start_date', 'r_time']).reset_index(drop=True)

# 2. 중복 제거 대상 설정 (실수/재결제 확률이 높은 기간제 상품들)
# 시간권이나 차감권은 하루에 여러 개 살 수 있으므로 제외합니다.
dedup_targets = ['무제한 기간권', '주말&야간패스', '매일 3시간패스']

# 3. 필터링 로직
# (유저ID, 결제일, 시작일, 카테고리)가 모두 같은데 'dedup_targets'에 속한다면 마지막 건만 남김
is_target = df['상품카테고리'].isin(dedup_targets)
duplicates = df[is_target].duplicated(subset=['client_uid', 'r_date', 'start_date', '상품카테고리'], keep='last')

# 중복된 인덱스만 드롭 (시간권/차감권은 이 과정에서 안전하게 보존됨)
df = df.drop(df[is_target][duplicates].index).reset_index(drop=True)

In [ ]:
df.head()

# 0. 사전분석 :  VIP로 가는 누적이용기간 확인

In [ ]:
import pandas as pd
import numpy as np

# 1. 환경 설정
analysis_end_date = pd.to_datetime('2025-12-31')

# 2. 날짜 형식 및 누적 이용일 계산
df['start_date'] = pd.to_datetime(df['start_date'])
df['actual_end_date'] = pd.to_datetime(df['actual_end_date'])
df = df.sort_values(['client_uid', 'start_date'])

# 건별 이용일수 및 유저별 누적 이용일수 계산
df['duration'] = (df['actual_end_date'] - df['start_date']).dt.days + 1
df['cum_duration'] = df.groupby('client_uid')['duration'].cumsum()

# 3. 누적 이용일 기준 재구매율 계산 함수
def calculate_retention_by_milestone(input_df, milestones=[30, 60, 90, 120, 150, 180]):
    results = []

    for m in milestones:
        # 조건 1: 누적 이용일(m일)을 최초로 돌파한 시점의 행 추출
        reached_m = input_df[input_df['cum_duration'] >= m].groupby('client_uid').head(1).copy()

        # 조건 2: 분석 기간(12/31) 이전에 해당 이용이 종료된 유저 (분모: 재구매 기회가 있었던 유저)
        finished_m = reached_m[reached_m['actual_end_date'] < analysis_end_date]

        total_users = len(finished_m)
        if total_users == 0: continue

        # 재구매 여부 확인: 해당 이용 종료일 이후에 결제 이력이 있는지 확인
        # (성능 최적화를 위해 merge 또는 join 방식 권장되나 기존 로직의 정확성을 유지함)
        repurchased_count = 0
        for _, row in finished_m.iterrows():
            uid = row['client_uid']
            end_dt = row['actual_end_date']

            has_next = input_df[(input_df['client_uid'] == uid) & (input_df['start_date'] > end_dt)]
            if not has_next.empty:
                repurchased_count += 1

        rate = (repurchased_count / total_users) * 100
        results.append({
            '누적_이용_기준': f'{m}일 이상',
            '대상_유저수(분모)': total_users,
            '재구매_유저수(분자)': repurchased_count,
            '재구매율(%)': round(rate, 2)
        })

    return pd.DataFrame(results)

# 4. 결과 실행
retention_report = calculate_retention_by_milestone(df)

# 5. 가독성 있게 출력
print("■ 누적 이용일수별 재구매율 분석 (120일 기준 근거)")
retention_report

In [ ]:
import pandas as pd
from datetime import datetime

# 1. 환경 설정
today = pd.to_datetime('2025-12-31')
migration_date = pd.to_datetime('2025-06-11')

# 2. 날짜 형식 및 기본 계산
df['start_date'] = pd.to_datetime(df['start_date'])
df['actual_end_date'] = pd.to_datetime(df['actual_end_date'])
df = df.sort_values(['client_uid', 'start_date'])
df['duration'] = (df['actual_end_date'] - df['start_date']).dt.days + 1
df['cum_duration'] = df.groupby('client_uid')['duration'].cumsum()

# 3. [핵심] 유저 타입 분류 (6/11 이전 가입 여부)
user_first_visit = df.groupby('client_uid')['start_date'].min()
migrated_uids = user_first_visit[user_first_visit < migration_date].index
df['user_type'] = df['client_uid'].apply(lambda x: 'Migrated' if x in migrated_uids else 'Pure_New')

# 4. 재구매율 계산 함수 (유저 타입별 필터 기능 추가)
def calculate_pure_retention_v2(input_df, user_type_label, milestones=[30, 60, 90, 120, 150, 180]):
    results = []
    # 특정 유저 타입만 필터링
    target_df = input_df[input_df['user_type'] == user_type_label].copy()

    for m in milestones:
        # 조건 1: 해당 누적 기간(m일) 돌파 행
        reached_m = target_df[target_df['cum_duration'] >= m].groupby('client_uid').head(1)

        # 조건 2: 12/31 이전에 이용이 종료된 건 (분모)
        finished_m = reached_m[reached_m['actual_end_date'] < today]

        total_users = len(finished_m)
        if total_users == 0: continue

        # 재구매 여부 확인
        repurchased_count = 0
        for _, row in finished_m.iterrows():
            uid = row['client_uid']
            end_dt = row['actual_end_date']
            # 종료일 이후에 다음 결제 기록이 있는지 전체 df에서 확인
            has_next = input_df[(input_df['client_uid'] == uid) & (input_df['start_date'] > end_dt)]
            if not has_next.empty:
                repurchased_count += 1

        rate = (repurchased_count / total_users) * 100
        results.append({
            '구분': user_type_label,
            '누적_이용_기준': f'{m}일',
            '대상_유저수': total_users,
            '재구매_유저수': repurchased_count,
            '재구매율(%)': round(rate, 2)
        })
    return pd.DataFrame(results)

# 5. 결과 실행 및 통합
report_migrated = calculate_pure_retention_v2(df, 'Migrated')
report_pure_new = calculate_pure_retention_v2(df, 'Pure_New')

# 두 결과를 합쳐서 가독성 있게 출력
final_report = pd.concat([report_migrated, report_pure_new])
final_report.set_index(['구분', '누적_이용_기준'])

In [ ]:
import pandas as pd
import numpy as np

# 1. 날짜 데이터 형식 확인 및 이용 일수 계산
df['start_date'] = pd.to_datetime(df['start_date'])
df['actual_end_date'] = pd.to_datetime(df['actual_end_date'])
df['usage_days'] = (df['actual_end_date'] - df['start_date']).dt.days + 1

# 2. [보조 컬럼] 무제한 기간권의 이용 일수만 따로 계산
df['only_period_days'] = np.where(df['상품카테고리'] == '무제한 기간권', df['usage_days'], 0)

# 3. 유저별 집계 (user_summary 생성)
user_summary = df.groupby('client_uid').agg(
    first_date=('start_date', 'min'),
    total_revenue=('actual_price', 'sum'),
    total_buy_count=('client_uid', 'count'),
    period_pass_count=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_pass_count=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    single_pass_count=('상품카테고리', lambda x: (x == '시간/1회권').sum()),
    total_period_days=('usage_days', 'sum') # 기간권 누적 이용일
).reset_index()

# 4. 유저 타입 분류 (6/11 개편일 기준)
migration_date = pd.to_datetime('2025-06-11')
user_summary['user_type'] = np.where(user_summary['first_date'] < migration_date, 'Migrated', 'Pure_New')

# 5. [결과] 그룹별 누적 이용 기간 통계 비교
usage_comparison = user_summary.groupby('user_type')['total_period_days'].agg([
    ('평균 이용일', 'mean'),
    ('중앙값(Median)', 'median'),
    ('최대 이용일', 'max'),
    ('유저 수', 'count')
]).round(1)

# 가독성 좋게 출력
usage_comparison

#1.코호트분류

In [ ]:
import pandas as pd
import numpy as np

# 1. 날짜 데이터 및 기본 전처리
df['start_date'] = pd.to_datetime(df['start_date'])
df['actual_end_date'] = pd.to_datetime(df['actual_end_date'])

# 2. [필터링] 6/11 이후 시작하고, 12/31 이전에 이용이 '종료'된 건만 추출
df_clean = df[
    (df['start_date'] >= '2025-06-11') &
    (df['actual_end_date'] <= '2025-12-31')
].copy()

# 3. 보조 컬럼 생성 (이용 일수 계산)
# [전체 이용일] 모든 상품의 이용 기간 (1회권/차감권 포함)
df_clean['usage_days'] = (df_clean['actual_end_date'] - df_clean['start_date']).dt.days + 1

# [기간권 전용일] 무제한 기간권일 때만 일수 집계 (분류용)
df_clean['only_period_days'] = np.where(df_clean['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']), df_clean['usage_days'], 0)

# 4. 유저별 집계 (Aggregation)
user_summary = df_clean.groupby('client_uid').agg(
    first_date=('start_date', 'min'),
    first_product_cat=('상품카테고리', 'first'),
    total_revenue=('actual_price', 'sum'),
    total_buy_count=('client_uid', 'count'),
    period_pass_count=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_pass_count=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    single_pass_count=('상품카테고리', lambda x: (x == '시간/1회권').sum()),

    # 코호트 분류를 위한 '기간권 누적일'
    total_period_days=('usage_days', 'sum'),

    # 리포트에서 "평균 이용일"로 보여줄 '전체 실제 이용일'
    total_actual_usage=('usage_days', 'sum')
).reset_index()
def finalize_cohorts_v8(row):
    # 1. VIP (기간권 합산 이용 120일 이상)
    if row['total_period_days'] >= 120:
        return '1. [정착] 4개월+ VIP'

    # 2. 기간권(무제한/주야) 2회 이상 구매
    if row['period_pass_count'] >= 2:
        return '2. [안착] 기간권 재구매'

    # 3. 기간권(무제한/주야) 첫 전환 (업셀링)
    if row['period_pass_count'] == 1:
        if row['first_product_cat'] in ['시간/1회권', '차감권(N회)']:
            return '3. [진입] 기간권 첫 전환(Up-sell)'
        else:
            # 처음부터 기간권으로 들어온 경우 아래 6번으로 통합 관리를 위해 유지
            return '6. [이탈] 고관여 단발성 구매'

    # 4. 차감권(N회) 단골 (기간권 0회 & 차감권 2회 이상)
    if row['n_pass_count'] >= 2:
        return '4. [실속] 차감권 단골'

    # 5. 1회권 반복 (기간권 0회 & 1회권 3회 이상)
    if row['single_pass_count'] >= 3:
        return '5. [간보기] 1회권 반복'

    # [수정된 구간] 6. 고관여 단발성 구매 (기간권 1회 OR 차감권 1회)
    # 기간권 1회는 위에서 처리되었고, 여기서는 '차감권만 딱 1회' 산 유저를 포함합니다.
    if row['period_pass_count'] == 0 and row['n_pass_count'] == 1:
        return '6. [이탈] 고관여 단발성 구매'

    # 7. 나머지 (1회권 1~2회 구매자만 남음)
    return '7. [신규] 단순 체험'

# 코호트 적용
user_summary['cohort'] = user_summary.apply(finalize_cohorts_v8, axis=1)

# 최종 요약 테이블 생성
cohort_report = user_summary.groupby('cohort').agg(
    유저수=('client_uid', 'count'),
    총매출=('total_revenue', 'sum'),
    인당평균매출_LTV=('total_revenue', 'mean'),
    평균실제이용일=('total_actual_usage', 'mean')
).reset_index()

cohort_report['매출비중(%)'] = (cohort_report['총매출'] / cohort_report['총매출'].sum() * 100).round(1)
cohort_report.sort_values('cohort', inplace=True)

## 6/11 최초 계약 시작유저만 filter

In [ ]:
import pandas as pd
import numpy as np

# 1. 날짜 데이터 및 기본 전처리
df['start_date'] = pd.to_datetime(df['start_date'])
df['actual_end_date'] = pd.to_datetime(df['actual_end_date'])

# [추가] 순수 신규 유저 식별: 전체 데이터에서 유저별 '생애 첫 시작일' 계산
user_first_ever = df.groupby('client_uid')['start_date'].min().reset_index()
user_first_ever.columns = ['client_uid', 'global_first_start']

# 2. [필터링] 6/11 이후 유입된 '순수 신규 유저'만 추출
pure_new_user_ids = user_first_ever[user_first_ever['global_first_start'] >= '2025-06-11']['client_uid']

df_clean = df[
    (df['client_uid'].isin(pure_new_user_ids)) &
    (df['start_date'] >= '2025-06-11') &
    (df['actual_end_date'] <= '2025-12-31')
].copy()

# 3. 보조 컬럼 생성 (이용 일수 계산)
df_clean['usage_days'] = (df_clean['actual_end_date'] - df_clean['start_date']).dt.days + 1
df_clean['only_period_days'] = np.where(df_clean['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']), df_clean['usage_days'], 0)

# 4. 유저별 집계 (Aggregation)
user_summary = df_clean.groupby('client_uid').agg(
    first_date=('start_date', 'min'),
    first_product_cat=('상품카테고리', 'first'),
    total_revenue=('actual_price', 'sum'),
    total_buy_count=('client_uid', 'count'), # 결제 횟수
    period_pass_count=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_pass_count=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    single_pass_count=('상품카테고리', lambda x: (x == '시간/1회권').sum()),
    total_period_days=('only_period_days', 'sum'),
    total_actual_usage=('usage_days', 'sum') # 전체 이용 일수
).reset_index()

# 5. 코호트 분류 함수 (V8)
def finalize_cohorts_v8(row):
    if row['total_period_days'] >= 120: return '1. [정착] 4개월+ VIP'
    if row['period_pass_count'] >= 2: return '2. [안착] 기간권 재구매'
    if row['period_pass_count'] == 1:
        if row['first_product_cat'] in ['시간/1회권', '차감권(N회)']:
            return '3. [진입] 기간권 첫 전환(Up-sell)'
        else:
            return '6. [이탈] 고관여 단발성 구매'
    if row['n_pass_count'] >= 2: return '4. [실속] 차감권 단골'
    if row['single_pass_count'] >= 3: return '5. [간보기] 1회권 반복'
    if row['period_pass_count'] == 0 and row['n_pass_count'] == 1: return '6. [이탈] 고관여 단발성 구매'
    return '7. [신규] 단순 체험'

user_summary['cohort'] = user_summary.apply(finalize_cohorts_v8, axis=1)

# 6. 최종 요약 테이블 생성 (평균 결제 횟수 및 이용 중앙값 추가)
cohort_report = user_summary.groupby('cohort').agg(
    유저수=('client_uid', 'count'),
    총매출=('total_revenue', 'sum'),
    인당평균매출_LTV=('total_revenue', 'mean'),
    인당평균결제횟수=('total_buy_count', 'mean'), # 인사이트용 지표 1
    평균실제이용일=('total_actual_usage', 'mean'),
    이용일중앙값=('total_actual_usage', 'median')  # 인사이트용 지표 2
).reset_index()

cohort_report['매출비중(%)'] = (cohort_report['총매출'] / cohort_report['총매출'].sum() * 100).round(1)
cohort_report.sort_values('cohort', inplace=True)

# 금액 가독성을 위한 포맷팅 (필요 시)
# cohort_report['인당평균매출_LTV'] = cohort_report['인당평균매출_LTV'].map('{:,.0f}원'.format)

cohort_report

In [ ]:
# 1. user_summary에 생성된 코호트 정보를 df_clean에 매핑
# client_uid를 기준으로 cohort 컬럼을 가져옵니다.
df_clean_cohort = df_clean.merge(
    user_summary[['client_uid', 'cohort']],
    on='client_uid',
    how='left'
)

# 2. '7. [신규] 단순 체험' 그룹만 필터링하여 Raw 데이터 확인
# 이용일수(usage_days)가 높은 순으로 정렬해서 "누가 평균을 올리고 있는지" 찾아봅니다.
target_raw = df_clean_cohort[df_clean_cohort['cohort'] == '6. [이탈] 고관여 단발성 구매'].sort_values(by='usage_days', ascending=False)

# 3. 결과 출력 (상위 20개 확인)
print("■ '신규 단순 체험' 그룹의 이용 기간 상위 유저 Raw 데이터")
display(target_raw[['client_uid', 'product_name', '상품카테고리', 'start_date', 'actual_end_date', 'usage_days']].head(20))

In [ ]:
cohort_report

# 2. 잔존유저 확인

In [ ]:


# 1. 기준일 및 7일 윈도우 설정
end_date = pd.to_datetime('2026-02-19')
start_date_window = end_date - pd.Timedelta(days=6)


# 6. 최근 7일 액티브 유저 식별 (전체 df 기준)
active_uids = df[
    (df['actual_end_date'] >= start_date_window) &
    (df['start_date'] <= end_date)
]['client_uid'].unique()

# 7. 액티브 상태 업데이트
user_summary['is_active'] = user_summary['client_uid'].isin(active_uids)

# 8. 최종 리포트 생성
active_7d_report = user_summary.groupby('cohort').agg(
    전체_유저수=('client_uid', 'count'),
    최근7일_액티브유저수=('is_active', 'sum'),
    평균_LTV=('total_revenue', 'mean')
).reset_index()

active_7d_report['생존율(%)'] = (active_7d_report['최근7일_액티브유저수'] / active_7d_report['전체_유저수'] * 100).round(1)
active_7d_report['평균_LTV'] = active_7d_report['평균_LTV'].map('{:,.0f}원'.format)
active_7d_report.sort_values('cohort', inplace=True)

print(f"■ 순수 신규 유저(6/11 이후 가입) 코호트별 최근 7일 생존 및 가치 현황")
active_7d_report

In [ ]:
import pandas as pd

# 1. 가입월 컬럼 생성
user_summary['start_month'] = user_summary['first_date'].dt.strftime('%m월')

# 2. 가입월별 [현재 생존자수] 피벗 테이블
pivot_active = user_summary.pivot_table(
    index='cohort',
    columns='start_month',
    values='is_active',
    aggfunc='sum'
).fillna(0).astype(int)

# 3. 가입월별 [전체 유입 유저수] 피벗 테이블
pivot_total = user_summary.pivot_table(
    index='cohort',
    columns='start_month',
    values='client_uid',
    aggfunc='count'
).fillna(0).astype(int)

# 4. 두 테이블을 결합하여 "생존 / 전체" 포맷으로 변환
# 데이터가 없는 구간은 '-'로 표시
final_count_pivot = (pivot_active.astype(str) + " / " + pivot_total.astype(str))
final_count_pivot = final_count_pivot.replace("0 / 0", "-")

# 5. 행별 전체 합계(전체 모수) 추가
total_stats = user_summary.groupby('cohort').agg(
    전체_모수=('client_uid', 'count'),
    전체_생존=('is_active', 'sum')
)
total_stats_str = total_stats['전체_생존'].astype(str) + " / " + total_stats['전체_모수'].astype(str)

# 6. 최종 리포트 결합
final_pivot_report = pd.concat([total_stats_str.rename('전체(생존/모수)'), final_count_pivot], axis=1).reset_index()

print(f"■ 가입월별 코호트 현재 생존 유저 현황 (현재 생존자 / 전체 유입)")
final_pivot_report

In [ ]:
# 1. 차감권을 한 번이라도 구매한 유저 식별
deduction_uids = df[df['상품카테고리'] == '차감권(N회)']['client_uid'].unique()

# 2. 해당 유저들의 전체 구매 이력 정렬
df_deduction_path = df[df['client_uid'].isin(deduction_uids)].sort_values(['client_uid', 'start_date'])

# 3. 유저별 '차감권 구매 시점' 찾기 (가장 첫 차감권 구매일)
first_deduction_date = df_deduction_path[df_deduction_path['상품카테고리'] == '차감권(N회)'].groupby('client_uid')['start_date'].min().reset_index()
first_deduction_date.columns = ['client_uid', 'deduction_start_date']

# 4. 차감권 구매 이후에 '기간권'을 구매했는지 확인
# 기간권 카테고리: ['무제한 기간권', '주말&야간패스']
df_with_first_deduction = df_deduction_path.merge(first_deduction_date, on='client_uid')

# 차감권 시작일 이후에 결제된 기간권만 필터링
post_deduction_period_pass = df_with_first_deduction[
    (df_with_first_deduction['start_date'] > df_with_first_deduction['deduction_start_date']) &
    (df_with_first_deduction['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']))
]

# 5. 결과 집계
converted_uids = post_deduction_period_pass['client_uid'].unique()
total_deduction_users = len(deduction_uids)
total_converted_users = len(converted_uids)

print(f"■ 차감권 기반 업셀링(Up-sell) 분석 결과")
print(f"- 차감권을 경험한 전체 유저: {total_deduction_users}명")
print(f"- 차감권 이용 후 기간권으로 전환한 유저: {total_converted_users}명")
print(f"- 차감권 -> 기간권 전환율: {(total_converted_users / total_deduction_users * 100):.1f}%")

In [ ]:
import pandas as pd

# 1. 데이터 정렬 및 '다음 상품 카테고리' 컬럼 생성
df_sorted = df_clean.sort_values(['client_uid', 'start_date']).copy()
df_sorted['next_cat'] = df_sorted.groupby('client_uid')['상품카테고리'].shift(-1)

# 2. 기간권 카테고리 정의
period_cats = ['무제한 기간권', '주말&야간패스']

# 3. 분석 대상 카테고리 (1회권 vs 차감권)
target_sources = ['시간/1회권', '차감권(N회)']

# 4. 효율 계산
conversion_results = []

for cat in target_sources:
    # 해당 상품의 총 구매 건수 (분모)
    total_purchases = len(df_sorted[df_sorted['상품카테고리'] == cat])

    # 해당 상품 구매 후 '다음 결제'가 기간권인 건수 (분자)
    to_period_conversions = len(df_sorted[
        (df_sorted['상품카테고리'] == cat) &
        (df_sorted['next_cat'].isin(period_cats))
    ])

    conv_rate = round((to_period_conversions / total_purchases * 100), 2) if total_purchases > 0 else 0

    conversion_results.append({
        '기준 상품': cat,
        '총 구매 건수(A)': total_purchases,
        '기간권 전환 건수(B)': to_period_conversions,
        '전환 효율(B/A, %)': conv_rate
    })

# 5. 결과 출력
eff_report = pd.DataFrame(conversion_results)
print(f"■ [상품별 전환 효율] 기간권으로 가는 더 강력한 징검다리는?")
eff_report

# 3. 벤치마크 코호트 분석

In [ ]:
import pandas as pd

# 1. 판다스 출력 옵션 설정 (줄바꿈 방지 및 모든 컬럼 표시)
pd.set_option('display.max_columns', None)      # 모든 컬럼 출력
pd.set_option('display.expand_frame_repr', False) # 줄바꿈 안함
pd.set_option('display.max_rows', None)         # 모든 행 출력

# 2. 컬럼명 확정
prod_col = 'product_name' if 'product_name' in df_clean.columns else '상품명'

# 3. 유저별 '생애 첫 결제' 추출
first_purchase_raw = df_clean.sort_values(['client_uid', 'start_date']).groupby('client_uid').first().reset_index()

# 4. 코호트 정보 결합
user_first_prod = first_purchase_raw[['client_uid', prod_col]].merge(
    user_summary[['client_uid', 'cohort']],
    on='client_uid',
    how='left'
)
# 5. 코호트별 상품 분포 계산 (열 기준 정규화: normalize='columns')
# 행(Index)을 상품으로, 열(Columns)을 코호트로 설정하여 각 코호트가 100%가 되게 합니다.
first_prod_stats = pd.crosstab(
    user_first_prod[prod_col],
    user_first_prod['cohort'],
    normalize='columns'  # 각 코호트(열)의 합을 100%로 만듦
) * 100

# 6. 가독성을 위해 상위 비중 상품 순으로 정렬 (VIP 코호트 기준)
final_pivot_df = first_prod_stats.round(1).sort_values('1. [정착] 4개월+ VIP', ascending=False).reset_index()

# 7. 결과 출력
print("■ [코호트별 %] 각 코호트 유저의 첫 시작 상품 분포")
print(final_pivot_df.to_string(index=False))


# 4. 업셀링 유저 상품 및 전환 소요 기간 분석

In [ ]:
import pandas as pd
import numpy as np

# 1. 3번 코호트 유저 ID 추출
target_cohort = '3. [진입] 기간권 첫 전환(Up-sell)'
c3_uids = user_summary[user_summary['cohort'] == target_cohort]['client_uid']

# 2. 분석 대상 유저의 전체 이력 추출 및 정렬
c3_df = df_clean[df_clean['client_uid'].isin(c3_uids)].sort_values(['client_uid', 'start_date'])

# 3. [로직] 첫 번째 기간권 구매 건과 그 직전 건 찾기
def analyze_upsell_moment(group):
    # 기간권 구매 기록들
    period_passes = group[group['상품카테고리'] == '무제한 기간권']
    if period_passes.empty: return None

    # 첫 번째 기간권 행
    first_period = period_passes.iloc[0]
    first_period_idx = first_period.name

    # 그 직전 행 (업셀링 전 마지막 1회권/차감권)
    prev_idx = group.index.get_loc(first_period_idx) - 1
    if prev_idx < 0: return None # 만약 첫 구매가 기간권이면 제외 (로직상 3번은 이럴 리 없음)

    prev_order = group.iloc[prev_idx]

    # 간격 계산: (기간권 시작일) - (직전 이용권 종료일)
    gap = (first_period['start_date'] - prev_order['actual_end_date']).days

    return pd.Series({
        'upsell_product': first_period[prod_col],
        'gap_days': gap
    })

# 4. 유저별 업셀링 모먼트 분석 적용
upsell_analysis = c3_df.groupby('client_uid').apply(analyze_upsell_moment).dropna()

# 5. 결과 집계 (어떤 상품을 샀나?)
product_dist = upsell_analysis['upsell_product'].value_counts(normalize=True).mul(100).round(1)

# 6. 결과 집계 (며칠 만에 샀나?)
gap_stats = upsell_analysis['gap_days'].describe(percentiles=[.25, .5, .75])

print("■ 3번 코호트: 첫 기간권 전환 시 선택한 상품 Top 5 (%)")
print(product_dist.head(5))

print("\n■ 3번 코호트: 직전 이용권 종료 후 기간권 구매까지 걸린 일수")
print(f"- 평균: {gap_stats['mean']:.1f}일")
print(f"- 중앙값(Median): {gap_stats['50%']:.0f}일")
print(f"- 75% 유저의 전환 주기: {gap_stats['75%']:.0f}일 이내")

In [ ]:
# 1. 분석 대상 코호트 필터링 (1, 2, 3번 포함하여 비교)
target_cohorts = ['1. [정착] 4개월+ VIP', '2. [안착] 기간권 재구매', '3. [진입] 기간권 첫 전환(Up-sell)']
df_growth = user_summary[user_summary['cohort'].isin(target_cohorts)].copy()

# 2. 시작 상품 카테고리 단순화 (기간권 스타트 vs 1회권/차감권 스타트)
def simplify_first_cat(cat):
    if cat in ['무제한 기간권', '주말&야간패스']:
        return '기간권 스타트'
    elif cat in ['시간/1회권', '차감권(N회)']:
        return '1회권/차감권 스타트'
    else:
        return '기타'

df_growth['start_type'] = df_growth['first_product_cat'].apply(simplify_first_cat)

# 3. 코호트별 시작 유형 분포 (Cross-tab)
growth_analysis = pd.crosstab(df_growth['cohort'], df_growth['start_type'], normalize='index') * 100
growth_analysis_count = pd.crosstab(df_growth['cohort'], df_growth['start_type'])

# 4. 가독성 있게 합치기
growth_report = pd.DataFrame({
    '전체 유저수': growth_analysis_count.sum(axis=1),
    '기간권 시작(%)': growth_analysis['기간권 스타트'].round(1),
    '1회권/차감권 시작(%)': growth_analysis['1회권/차감권 스타트'].round(1)
})

print("■ 고관여 코호트의 최초 진입 상품 분포")
print(growth_report)

In [ ]:
# 1. 분석 대상 코호트 필터링 (순수 신규 유저 중 고관여 집단만)
target_cohorts = ['1. [정착] 4개월+ VIP', '2. [안착] 기간권 재구매', '3. [진입] 기간권 첫 전환(Up-sell)']
df_growth = user_summary[user_summary['cohort'].isin(target_cohorts)].copy()

# 2. 시작 상품 카테고리 단순화
def simplify_first_cat(cat):
    if cat in ['무제한 기간권', '주말&야간패스']:
        return '기간권 스타트'
    elif cat in ['시간/1회권', '차감권(N회)']:
        return '1회권/차감권 스타트'
    else:
        return '기타'

df_growth['start_type'] = df_growth['first_product_cat'].apply(simplify_first_cat)

# 3. 분석: 코호트별 시작 유형 비중 계산
growth_analysis = pd.crosstab(df_growth['cohort'], df_growth['start_type'], normalize='index') * 100
growth_analysis_count = pd.crosstab(df_growth['cohort'], df_growth['start_type'])

# 4. 리포트 생성 (가독성 강화)
growth_report = pd.DataFrame({
    '총 유저수': growth_analysis_count.sum(axis=1),
    '업셀링(1회권 시작) 유저수': growth_analysis_count['1회권/차감권 스타트'],
    '업셀링 성공 비중(%)': growth_analysis['1회권/차감권 스타트'].round(1),
    '다이렉트(기간권 시작) 비중(%)': growth_analysis['기간권 스타트'].round(1)
}).reset_index()

print("■ 고관여 코호트의 진입 경로 분석 (순수 신규 기준)")
growth_report

In [ ]:
import pandas as pd

# 1. 유저별 첫 결제 데이터 및 코호트 정보 준비
first_orders = df.sort_values(['client_uid', 'start_date', 'r_time']).groupby('client_uid').head(1)
first_orders_with_cohort = first_orders.merge(
    user_summary[['client_uid', 'cohort']],
    on='client_uid',
    how='inner'
)

# 2. 상세 상품 카테고리 분류 (알려주신 '무제한 - 1일' 반영)
def categorize_experience_prod_v3(name):
    if '무제한 - 1일' in name:
        return '3. [현재] 1일권 (무제한-1일)'
    elif '24시간' in name:
        return '2. [과거] 24시간패스'
    elif '시간' in name and '패스' in name:
        return '1. [과거] 시간패스(2~12H)'
    else:
        return '기타'

first_orders_with_cohort['exp_group'] = first_orders_with_cohort['product_name'].apply(categorize_experience_prod_v3)
exp_targets = first_orders_with_cohort[first_orders_with_cohort['exp_group'] != '기타'].copy()

# 3. 성과 데이터 결합
comparison_df = exp_targets.merge(
    user_summary[['client_uid', 'total_revenue', 'total_period_days']],
    on='client_uid',
    how='inner'
)

# 4. 그룹별 최종 집계 (성공 지표에 3번 코호트 포함)
exp_analysis_v2 = comparison_df.groupby('exp_group').agg(
    유저수=('client_uid', 'count'),
    평균_LTV=('total_revenue', 'mean'),
    평균_총이용일=('total_period_days', 'mean'),
    # 1, 2, 3번 코호트(정착/안착/진입) 합산 비중
    전체_전환율_123번=('cohort', lambda x: (x.isin(['1. [정착] 4개월+ VIP', '2. [안착] 기간권 재구매', '3. [진입] 기간권 첫 전환(Up-sell)']).sum() / len(x) * 100)),
    # 그 중 3번 코호트만 따로 보기
    진입_비중_3번_only=('cohort', lambda x: (x == '3. [진입] 기간권 첫 전환(Up-sell)').sum() / len(x) * 100)
).reset_index()

# 5. 결과 포맷팅
exp_analysis_v2['평균_LTV'] = exp_analysis_v2['평균_LTV'].map('{:,.0f}원'.format)
exp_analysis_v2['전체_전환율_123번'] = exp_analysis_v2['전체_전환율_123번'].round(1).astype(str) + '%'
exp_analysis_v2['진입_비중_3번_only'] = exp_analysis_v2['진입_비중_3번_only'].round(1).astype(str) + '%'

print("■ 체험 상품별 업셀링 및 전체 전환 성과 (3번 코호트 포함)")
exp_analysis_v2

In [ ]:
# 1. 전체 데이터의 시작과 끝 확인
print(f"■ 전체 데이터 시작일 (start_date min): {df['start_date'].min()}")
print(f"■ 전체 데이터 종료일 (actual_end_date max): {df['actual_end_date'].max()}")

# 2. '무제한 - 1일' 상품만 따로 기간 확인
day_pass_df = df[df['product_name'].str.contains('무제한 - 1일', na=False)]
if not day_pass_df.empty:
    print(f"\n■ '무제한 - 1일' 첫 결제일: {day_pass_df['start_date'].min()}")
    print(f"■ '무제한 - 1일' 마지막 종료일: {day_pass_df['actual_end_date'].max()}")
else:
    print("\n[알림] '무제한 - 1일' 상품 데이터가 없습니다.")

# 4. 재구매 주기 및 이탈 임계점(Churn Point) 분석

In [ ]:
import pandas as pd

# 1. 유저별 구매 간격 계산 (이전 종료일 ~ 다음 시작일)
df_gap = df.sort_values(['client_uid', 'start_date']).copy()
df_gap['next_start'] = df_gap.groupby('client_uid')['start_date'].shift(-1)
df_gap['re_purchase_gap'] = (df_gap['next_start'] - df_gap['actual_end_date']).dt.days

# 2. 재구매가 발생한 건만 추출 (gap이 0 이상인 경우)
re_purchase_data = df_gap[df_gap['re_purchase_gap'].notnull() & (df_gap['re_purchase_gap'] >= 0)]

# 3. 재구매 주기 분포 분석
gap_dist = re_purchase_data['re_purchase_gap'].value_counts().sort_index().cumsum()
gap_dist_pct = (gap_dist / len(re_purchase_data)) * 100

# 4. 주요 백분위 지점 찾기 (80%, 90%, 95%가 돌아오는 시점)
p80 = re_purchase_data['re_purchase_gap'].quantile(0.8)
p90 = re_purchase_data['re_purchase_gap'].quantile(0.9)
p95 = re_purchase_data['re_purchase_gap'].quantile(0.95)

print(f"■ [재구매 골든타임] 분석 결과")
print(f"- 재구매 유저의 80%는 종료 후 {p80:.0f}일 이내에 돌아옵니다.")
print(f"- 재구매 유저의 90%는 종료 후 {p90:.0f}일 이내에 돌아옵니다.")
print(f"- 재구매 유저의 95%는 종료 후 {p95:.0f}일 이내에 돌아옵니다.")
print("-" * 50)
print("■ [실질적 이탈] 1년(365일) 이내 복귀 비중")
long_return = (re_purchase_data['re_purchase_gap'] > 180).sum() / len(re_purchase_data) * 100
print(f"- 6개월(180일) 이후에 돌아온 '기적의 생존자' 비중: {long_return:.2f}%")

In [ ]:
re_purchase_data.sort_values('r_date').head()

In [ ]:
# 1~2단계는 동일 (생략)

# 3. 재구매 리드타임 구간(Bucket) 설정
bins = [0, 7, 14, 30, 60, 90, 180, 365, float('inf')]
labels = ['1주 이내', '2주 이내', '1개월 이내', '2개월 이내', '3개월 이내', '6개월 이내', '1년 이내', '1년 초과']

re_purchase_data['gap_bucket'] = pd.cut(re_purchase_data['re_purchase_gap'], bins=bins, labels=labels, right=False)

# 4. 구간별 통계 집계
gap_analysis = re_purchase_data.groupby('gap_bucket').size().reset_index(name='유저수')
gap_analysis['비중(%)'] = (gap_analysis['유저수'] / gap_analysis['유저수'].sum() * 100).round(1)
gap_analysis['누적비중(%)'] = gap_analysis['비중(%)'].cumsum().round(1)

# 5. 주요 Quantile(80%, 90%, 95%) 지점 추출용 테이블
quantiles = [0.8, 0.9, 0.95]
q_values = re_purchase_data['re_purchase_gap'].quantile(quantiles).values
q_report = pd.DataFrame({
    '구분': ['Golden Time (P80)', 'Danger Zone (P90)', 'Churn Limit (P95)'],
    '재구매_소요기간': [f'{int(q)}일 이내' for q in q_values],
    '누적_귀환율': ['80%', '90%', '95%']
})

print("■ [Table 1] 재구매 구간별 분포")
gap_analysis

In [ ]:
print("\n■ [Table 2] 재구매 확률 임계점 (Golden Time)")
q_report

#주말 & 야간패스 유저

In [ ]:
df_clean.head()

In [ ]:

df_clean['start_date'] = pd.to_datetime(df_clean['start_date'])
df_clean['actual_end_date'] = pd.to_datetime(df_clean['actual_end_date'])

# 2. 유저별 그룹화 및 지표 계산
# 각 유저가 '무제한 기간권' 또는 '주말&야간패스'를 한 번이라도 샀는지 체크
user_cats = df_clean.groupby('client_uid')['상품카테고리'].apply(set).reset_index()

user_cats['is_unlimited'] = user_cats['상품카테고리'].apply(lambda x: '무제한 기간권' in x)
user_cats['is_weekend_night'] = user_cats['상품카테고리'].apply(lambda x: '주말&야간패스' in x)

# 3. 그룹 라벨링
def assign_group(row):
    if row['is_unlimited'] and row['is_weekend_night']:
        return '둘 다 이용'
    if row['is_unlimited']:
        return '무제한 기간권 전용'
    if row['is_weekend_night']:
        return '주말&야간패스 전용'
    return '기타 상품군'

user_cats['user_group'] = user_cats.apply(assign_group, axis=1)

# 4. 전체 유저 통계 결합 (LTV, 결제횟수 등)
user_metrics = df_clean.groupby('client_uid').agg(
    total_revenue=('actual_price', 'sum'),
    total_buy_count=('client_uid', 'count'),
    total_usage_days=('actual_end_date', lambda x: ((x.max() - df_clean.loc[x.index, 'start_date'].min()).days + 1))
).reset_index()

final_analysis = user_metrics.merge(user_cats[['client_uid', 'user_group']], on='client_uid')

# 5. [결과 1] 카테고리 그룹별 LTV 및 주요 지표 비교
group_report = final_analysis.groupby('user_group').agg(
    유저수=('client_uid', 'count'),
    인당_LTV=('total_revenue', 'mean'),
    평균_결제횟수=('total_buy_count', 'mean'),
    평균_총이용일=('total_usage_days', 'mean')
).reset_index()

# 포맷팅
group_report['인당_LTV'] = group_report['인당_LTV'].map('{:,.0f}원'.format)
group_report['평균_총이용일'] = group_report['평균_총이용일'].round(1)

print("■ 상품 카테고리 그룹별 LTV 비교")
print(group_report.to_string(index=False))

# 6. [결과 2] 코호트와 카테고리의 관계 (만약 cohort 컬럼이 생성되어 있다면)
# user_summary와 조인하여 확인 가능
if 'cohort' in locals() or 'user_summary' in globals():
    # user_summary에 이미 생성된 cohort 정보를 가져와서 cross-tab
    cohort_mapping = user_summary[['client_uid', 'cohort']].merge(user_cats[['client_uid', 'user_group']], on='client_uid')
    cohort_dist = pd.crosstab(cohort_mapping['user_group'], cohort_mapping['cohort'], normalize='index') * 100

    print("\n■ 그룹별 코호트 분포 (%)")
cohort_dist

In [ ]:
# 6. [결과 2] 코호트와 카테고리의 관계 (유저수 기준)
if 'cohort' in locals() or 'user_summary' in globals():
    # user_summary의 cohort 정보와 user_cats의 그룹 정보를 결합
    cohort_mapping = user_summary[['client_uid', 'cohort']].merge(user_cats[['client_uid', 'user_group']], on='client_uid')

    # normalize를 제거하여 비율이 아닌 '실제 유저수'를 집계
    cohort_dist = pd.crosstab(
        cohort_mapping['user_group'],
        cohort_mapping['cohort'],
        margins=True,       # 우측과 하단에 '합계(All)' 행/열을 추가하여 보기 편하게 만듭니다
        margins_name='합계'
    )

    print("\n■ 그룹별 코호트 유저수 분포 (단위: 명)")
cohort_dist

In [ ]:
group_report

In [ ]:
# 1. '둘 다 이용' 그룹에 해당하는 유저 리스트 추출
both_users_uids = user_cats[user_cats['user_group'] == '둘 다 이용']['client_uid']

# 2. 해당 유저들의 '무제한 기간권'과 '주말&야간패스' 결제 내역만 필터링
df_both = df[
    (df['client_uid'].isin(both_users_uids)) &
    (df['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']))
].copy()

# 3. 날짜 형식 확정 및 유저별 상품별 '최초 구매일' 계산
df_both['start_date'] = pd.to_datetime(df_both['start_date'])
first_purchase_dates = df_both.groupby(['client_uid', '상품카테고리'])['start_date'].min().unstack()

# 4. 이동 방향 판별 함수
def analyze_flow(row):
    unlimited = row['무제한 기간권']
    weekend = row['주말&야간패스']

    if pd.isna(unlimited) or pd.isna(weekend): return "데이터 확인 필요"
    if unlimited < weekend:
        return '무제한 → 주말&야간 (다운그레이드/안착)'
    elif weekend < unlimited:
        return '주말&야간 → 무제한 (업셀링/성장)'
    else:
        return '동시/같은 날 시작'

first_purchase_dates['flow'] = first_purchase_dates.apply(analyze_flow, axis=1)

# 5. 결과 요약
flow_summary = first_purchase_dates['flow'].value_counts()
flow_pct = (first_purchase_dates['flow'].value_counts(normalize=True) * 100).round(1)

print("■ '둘 다 이용' 유저의 상품 간 이동 경로")
print(pd.concat([flow_summary, flow_pct], axis=1, keys=['유저수', '비중(%)']))

In [ ]:
cohort_mapping.head()

#구매 시작시기별 코호트

In [ ]:
import pandas as pd
import numpy as np

# 1. 날짜 처리 (전체 df 대상)
df['start_date'] = pd.to_datetime(df['start_date'])
df['actual_end_date'] = pd.to_datetime(df['actual_end_date'])

# 2. 유저별 '생애 첫 계약일' 찾기
user_first_ever = df.groupby('client_uid')['start_date'].min().reset_index()
user_first_ever.columns = ['client_uid', 'first_ever_date']

# 3. 6월~7월 진입 유저 리스트 추출
june_july_joiners = user_first_ever[
    (user_first_ever['first_ever_date'] >= '2025-06-01') &
    (user_first_ever['first_ever_date'] <= '2025-07-31')
]['client_uid'].unique()

# 4. 분석 대상 데이터 필터링 (6~7월 진입자의 2025년 전체 활동)
df_target = df[
    (df['client_uid'].isin(june_july_joiners)) &
    (df['start_date'] >= '2025-06-11') &
    (df['actual_end_date'] <= '2025-12-31')
].copy()

df_target['usage_days'] = (df_target['actual_end_date'] - df_target['start_date']).dt.days + 1

# 5. 유저별 집계
user_summary_jj = df_target.groupby('client_uid').agg(
    first_product_cat=('상품카테고리', 'first'),
    period_pass_count=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_pass_count=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    single_pass_count=('상품카테고리', lambda x: (x == '시간/1회권').sum()),
    total_revenue=('actual_price', 'sum'),
    total_period_days=('usage_days', 'sum')
).reset_index()

# 6. v8 코호트 로직 적용
def finalize_cohorts_v8(row):
    if row['total_period_days'] >= 120: return '1. [정착] 4개월+ VIP'
    if row['period_pass_count'] >= 2:    return '2. [안착] 기간권 재구매'
    if row['period_pass_count'] == 1:
        if row['first_product_cat'] in ['시간/1회권', '차감권(N회)']:
            return '3. [진입] 기간권 첫 전환(Up-sell)'
        else:
            return '6. [이탈] 고관여 단발성 구매'
    if row['n_pass_count'] >= 2:         return '4. [실속] 차감권 단골'
    if row['single_pass_count'] >= 3:    return '5. [간보기] 1회권 반복'
    if row['period_pass_count'] == 0 and row['n_pass_count'] == 1:
        return '6. [이탈] 고관여 단발성 구매'
    return '7. [신규] 단순 체험'

user_summary_jj['cohort'] = user_summary_jj.apply(finalize_cohorts_v8, axis=1)

# 7. 6-7월 진입자 전용 리포트 생성
jj_report = user_summary_jj.groupby('cohort').agg(
    유저수=('client_uid', 'count'),
    평균_LTV=('total_revenue', 'mean'),
    평균_누적이용일=('total_period_days', 'mean')
).reset_index()

jj_report['평균_LTV'] = jj_report['평균_LTV'].map('{:,.0f}원'.format)
jj_report['평균_누적이용일'] = jj_report['평균_누적이용일'].round(1)

print("■ 25년 6~7월 신규 유저의 연말(12/31) 기준 코호트 성적표")
print(jj_report.sort_values('cohort'))

In [ ]:
jj_report

#체크인 기록 분석

In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_check_in = pd.read_sql("""
                               SELECT
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS check_in_date,
                                        ch.check_in_history_uid,
                                        ci.contract_participant_uid,
                                        c.contract_uid,
                                        cp.client_uid,
                                        ch.branch_uid,
                                        b.street_address,
                                        b.sub_type,
                                        b.display_name,
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS check_in_Time,
                                        TO_CHAR(ch.end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS check_out_time
                                FROM check_in ci
                                left join check_in_history ch
                                on ci.check_in_uid = ch.check_in_uid
                                left join branch b
                                on ch.branch_uid  = b.branch_uid
                                left join contract_participant cp
                                on cp.contract_participant_uid = ci.contract_participant_uid
                                left join contract c
                                on c.contract_uid = cp.contract_uid
                                LEFT JOIN price_policy pp
                                ON c.price_policy_uid = pp.price_policy_uid
                                where c.actual_price > 0

                    """, conn)
    conn.close()
df_check_in.head()

In [ ]:
# 1. 분석 대상 기간 설정
analysis_start = '2025-06-11'
analysis_end = '2025-12-31'

# 2. 해당 기간 내에 '첫 결제'를 한 신규 유저만 추출 (진정한 하반기 코호트)
new_joiners_after_june = df.groupby('client_uid')['start_date'].min().reset_index()
new_joiners_after_june = new_joiners_after_june[
    (new_joiners_after_june['start_date'] >= analysis_start) &
    (new_joiners_after_june['start_date'] <= analysis_end)
]['client_uid']

# 3. user_summary에서 이 유저들만 필터링
user_summary_v2 = user_summary[user_summary['client_uid'].isin(new_joiners_after_june)].copy()

# 4. 필터링된 코호트 현황 확인
cohort_filtered = user_summary_v2.groupby('cohort').agg(
    유저수=('client_uid', 'count'),
    평균_LTV=('total_revenue', 'mean'),
    평균_총이용일=('total_period_days', 'mean')
).reset_index()

print(f"■ {analysis_start} 이후 신규 진입자 기준 코호트 현황")
display(cohort_filtered.sort_values('cohort'))

In [ ]:
import pandas as pd
import numpy as np

# 1. 분석 유니버스 및 날짜 고정 (2025-06-11 ~ 2025-12-31)
# Cell 25의 정밀 로직을 마스터로 선언합니다.
analysis_start = '2025-06-11'
analysis_end = '2025-12-31'

# 2. 순수 신규 유입자 식별 로직
user_first_ever = df.groupby('client_uid')['start_date'].min().reset_index()
user_first_ever.columns = ['client_uid', 'global_first_start']
pure_new_user_ids = user_first_ever[user_first_ever['global_first_start'] >= analysis_start]['client_uid']

# 3. 79명 VIP를 만드는 데이터셋(df_clean) 강제 재생성
df_clean = df[
    (df['client_uid'].isin(pure_new_user_ids)) &
    (df['start_date'] >= analysis_start) &
    (df['actual_end_date'] <= analysis_end)
].copy()

# 4. 일수 계산 보정 (+1일 처리 및 기간권 전용일 발라내기)
df_clean['usage_days'] = (df_clean['actual_end_date'] - df_clean['start_date']).dt.days + 1
df_clean['only_period_days'] = np.where(df_clean['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']), df_clean['usage_days'], 0)

# 5. [마스터] user_summary 확정
user_summary = df_clean.groupby('client_uid').agg(
    first_date=('start_date', 'min'),
    first_product_cat=('상품카테고리', 'first'),
    total_revenue=('actual_price', 'sum'),
    total_buy_count=('client_uid', 'count'),
    period_pass_count=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_pass_count=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    single_pass_count=('상품카테고리', lambda x: (x == '시간/1회권').sum()),
    total_period_days=('only_period_days', 'sum'), # 120일 VIP 판정의 핵심 (기간권만!)
    total_actual_usage=('usage_days', 'sum')       # 전체 이용일 (보고서용)
).reset_index()

# 6. 코호트 분류 함수 적용
def finalize_cohorts_v8_fixed(row):
    if row['total_period_days'] >= 120: return '1. [정착] 4개월+ VIP'
    if row['period_pass_count'] >= 2: return '2. [안착] 기간권 재구매'
    if row['period_pass_count'] == 1:
        if row['first_product_cat'] in ['시간/1회권', '차감권(N회)']: return '3. [진입] 기간권 첫 전환(Up-sell)'
        else: return '6. [이탈] 고관여 단발성 구매'
    if row['n_pass_count'] >= 2: return '4. [실속] 차감권 단골'
    if row['single_pass_count'] >= 3: return '5. [간보기] 1회권 반복'
    if row['period_pass_count'] == 0 and row['n_pass_count'] == 1: return '6. [이탈] 고관여 단발성 구매'
    return '7. [신규] 단순 체험'

user_summary['cohort'] = user_summary.apply(finalize_cohorts_v8_fixed, axis=1)

# 7. 검증 출력
print(f"✅ 코호트 유저수 확인 (1번 VIP): {user_summary[user_summary['cohort'] == '1. [정착] 4개월+ VIP'].shape[0]}명")

In [ ]:
import pandas as pd
import numpy as np

# 1. 분석 기준일 및 대상 유저(순수 신규) 식별
analysis_start, analysis_end = '2025-06-11', '2025-12-31'
user_first_ever = df.groupby('client_uid')['start_date'].min().reset_index()
pure_new_uids = user_first_ever[user_first_ever['start_date'] >= analysis_start]['client_uid']

# 2. 계약 데이터 유니버스 고정 및 일수 계산 (12/31 이전 종료 건 기준)
df_master = df[(df['client_uid'].isin(pure_new_uids)) &
               (df['start_date'] >= analysis_start) &
               (df['actual_end_date'] <= analysis_end)].copy()

df_master['usage_days'] = (pd.to_datetime(df_master['actual_end_date']) - pd.to_datetime(df_master['start_date'])).dt.days + 1
df_master['period_days'] = np.where(df_master['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']), df_master['usage_days'], 0)

# 3. 코호트 마스터 명단 생성 (79명 VIP 로직 적용)
user_summary_final = df_master.groupby('client_uid').agg(
    first_cat=('상품카테고리', 'first'),
    p_cnt=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_cnt=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    s_cnt=('상품카테고리', lambda x: (x == '시간/1회권').sum()),
    vip_days=('period_days', 'sum'),
    total_days=('usage_days', 'sum')
).reset_index()

def get_cohort(row):
    if row['vip_days'] >= 120: return '1. [정착] 4개월+ VIP'
    if row['p_cnt'] >= 2: return '2. [안착] 기간권 재구매'
    if row['p_cnt'] == 1:
        return '3. [진입] 기간권 첫 전환(Up-sell)' if row['first_cat'] in ['시간/1회권', '차감권(N회)'] else '6. [이탈] 고관여 단발성 구매'
    if row['n_cnt'] >= 2: return '4. [실속] 차감권 단골'
    if row['s_cnt'] >= 3: return '5. [간보기] 1회권 반복'
    if row['p_cnt'] == 0 and row['n_cnt'] == 1: return '6. [이탈] 고관여 단발성 구매'
    return '7. [신규] 단순 체험'

user_summary_final['cohort'] = user_summary_final.apply(get_cohort, axis=1)

# 4. 체크인 데이터 기간 제한 (12/31까지) 및 결합
df_check_in_2025 = df_check_in[pd.to_datetime(df_check_in['check_in_time']) <= analysis_end + ' 23:59:59'].copy()
user_summary_final['client_uid'] = user_summary_final['client_uid'].astype(str)
df_check_in_2025['client_uid'] = df_check_in_2025['client_uid'].astype(str)

merged = pd.merge(df_check_in_2025, user_summary_final[['client_uid', 'cohort', 'total_days']], on='client_uid', how='inner')

# 5. 최종 방문 밀도 산출
density_report = merged.groupby('cohort').agg(
    u_cnt=('client_uid', 'nunique'),
    total_v=('check_in_history_uid', 'count'),
    avg_days=('total_days', 'mean')
).reset_index()

density_report['방문_밀도(%)'] = ((density_report['total_v'] / density_report['u_cnt']) / density_report['avg_days'] * 100).round(1)

# 결과 반환
display(density_report[['cohort', '방문_밀도(%)']].sort_values('cohort'))

In [ ]:
import pandas as pd
import numpy as np

# 1. 분석 유니버스 고정 (2025-06-11 ~ 2025-12-31)
analysis_start, analysis_end = '2025-06-11', '2025-12-31'

# 2. 마스터 코호트 명단 확정 (79명 VIP 로직 강제 적용)
user_first_ever = df.groupby('client_uid')['start_date'].min().reset_index()
pure_new_uids = user_first_ever[user_first_ever['start_date'] >= analysis_start]['client_uid']

df_master = df[(df['client_uid'].isin(pure_new_uids)) &
               (df['start_date'] >= analysis_start) &
               (df['actual_end_date'] <= analysis_end)].copy()

df_master['usage_days'] = (pd.to_datetime(df_master['actual_end_date']) - pd.to_datetime(df_master['start_date'])).dt.days + 1
df_master['period_days_vip'] = np.where(df_master['상품카테고리'].isin(['무제한 기간권', '주말&야간패스']), df_master['usage_days'], 0)

user_summary_master = df_master.groupby('client_uid').agg(
    first_cat=('상품카테고리', 'first'),
    p_cnt=('상품카테고리', lambda x: x.str.contains('무제한|주말|야간', na=False).sum()),
    n_cnt=('상품카테고리', lambda x: (x == '차감권(N회)').sum()),
    s_cnt=('상품카테고리', lambda x: (x == '시간/1회권').sum()),
    vip_days=('period_days_vip', 'sum'),
    total_contract_days=('usage_days', 'sum')
).reset_index()

# 코호트 분류 (79명 버전)
user_summary_master['cohort'] = user_summary_master.apply(
    lambda r: '1. [정착] 4개월+ VIP' if r['vip_days'] >= 120 else (
              '2. [안착] 기간권 재구매' if r['p_cnt'] >= 2 else (
              '3. [진입] 기간권 첫 전환(Up-sell)' if r['p_cnt'] == 1 and r['first_cat'] in ['시간/1회권', '차감권(N회)'] else (
              '6. [이탈] 고관여 단발성 구매' if r['p_cnt'] == 1 else (
              '4. [실속] 차감권 단골' if r['n_cnt'] >= 2 else (
              '5. [간보기] 1회권 반복' if r['s_cnt'] >= 3 else (
              '6. [이탈] 고관여 단발성 구매' if r['n_cnt'] == 1 else '7. [신규] 단순 체험')))))), axis=1)

# 3. 체크인 데이터 기간 제한 및 중복 제거 (출근율 계산용)
df_check_in['check_in_time'] = pd.to_datetime(df_check_in['check_in_time'])
df_check_in_2025 = df_check_in[(df_check_in['check_in_time'] >= analysis_start) &
                              (df_check_in['check_in_time'] <= analysis_end + ' 23:59:59')].copy()

# 데이터 타입 일치
user_summary_master['client_uid'] = user_summary_master['client_uid'].astype(str)
df_check_in_2025['client_uid'] = df_check_in_2025['client_uid'].astype(str)

# 4. 행동 데이터 병합
merged = pd.merge(df_check_in_2025, user_summary_master[['client_uid', 'cohort', 'total_contract_days']], on='client_uid', how='inner')

# 5. [핵심] 코호트별 지표 산출
# - 방문 밀도: 총 입실 횟수 / 총 계약일 (N차 입실 포함)
# - 실제 출근율: 입실한 '날짜' 수 / 총 계약일 (하루 여러 번 와도 1일로 계산)
final_report = merged.groupby('cohort').agg(
    유저수=('client_uid', 'nunique'),
    총_입실횟수=('check_in_history_uid', 'count'),
    실제_출근일수=('check_in_date', 'nunique'),
    평균_계약일=('total_contract_days', 'mean')
).reset_index()

# 지표 계산
final_report['방문_밀도(%)'] = ((final_report['총_입실횟수'] / final_report['유저수']) / final_report['평균_계약일'] * 100).round(1)
final_report['순수_출근율(%)'] = ((final_report['실제_출근일수'] / final_report['유저수']) / final_report['평균_계약일'] * 100).round(1)

print("■ [검증 완료] 2025년 하반기 코호트별 이용 강도 분석")
display(final_report[['cohort', '유저수', '방문_밀도(%)', '순수_출근율(%)']].sort_values('cohort'))

In [ ]:
import pandas as pd

# 1. 3번 코호트 유저 리스트 추출
upsell_uids = user_summary[user_summary['cohort'] == '3. [진입] 기간권 첫 전환(Up-sell)']['client_uid']

# 2. 이 유저들의 결제 이력 중 '두 번째 결제(첫 업셀링)' 찾기
# (첫 번째는 1회권/1일권이었을 것이므로)
upsell_orders = df[df['client_uid'].isin(upsell_uids)].sort_values(['client_uid', 'start_date'])
first_upsell_order = upsell_orders.groupby('client_uid').nth(1) # 유저별 두 번째 결제건

# 3. 업셀링 상품 유형 분류
def classify_upsell_prod(name):
    if '주말' in name or '심야' in name or '저녁' in name:
        return '주말/야간권'
    elif '무제한' in name or 'Pass' in name or 'PASS' in name:
        return '전일 무제한권'
    elif '차감' in name or '횟수' in name:
        return '차감권/횟수권'
    else:
        return '기타/전용석'

first_upsell_order['upsell_type'] = first_upsell_order['product_name'].apply(classify_upsell_prod)

# 4. 결과 집계
upsell_dist = first_upsell_order['upsell_type'].value_counts(normalize=True) * 100
upsell_count = first_upsell_order['upsell_type'].value_counts()

upsell_report = pd.DataFrame({
    '전환_상품_유형': upsell_dist.index,
    '유저수': upsell_count.values,
    '비중(%)': upsell_dist.values.round(1)
})

print("■ 3번 코호트: 첫 업셀링 시 선택한 상품 비중")
display(upsell_report)

In [ ]:
first_upsell_order

In [ ]:
import pandas as pd

# 1. 7번 코호트 유저 리스트
cohort_7_uids = user_summary[user_summary['cohort'] == '7. [신규] 단순 체험']['client_uid']
df_c7 = df[df['client_uid'].isin(cohort_7_uids)].copy()

# 2. 금액 컬럼명 자동 매칭 (에러 방지)
# 'total_revenue'가 없으면 'price', 'amount', 'revenue' 중 있는 것을 찾습니다.
rev_col = [col for col in ['total_revenue', 'price', 'amount', 'revenue'] if col in df.columns]
rev_col = rev_col[0] if rev_col else None

# 3. 7번 그룹 내 체크인 데이터 분석
# (1일 투어 여부를 확인하기 위해 날짜별 지점 방문 수를 계산)
c7_checkin = df_check_in[df_check_in['client_uid'].isin(cohort_7_uids)].copy()
c7_checkin['check_in_date'] = c7_checkin['check_in_time'].dt.date

# 유저별/날짜별 방문 지점 수 계산
daily_tour = c7_checkin.groupby(['client_uid', 'check_in_date'])['branch_uid'].nunique().reset_index()
tour_users = daily_tour[daily_tour['branch_uid'] >= 2] # 하루에 2개 지점 이상 방문한 유저

# 4. 검증 결과 집계
print(f"■ 7번 코호트 상세 검증 (총 {len(cohort_7_uids)}명)")

# A. 매출 기반 검증
if rev_col:
    free_users = df_c7.groupby('client_uid')[rev_col].sum()
    free_count = len(free_users[free_users == 0])
    print(f"1. 결제 금액 0원(무료권) 유저: {free_count}명")
else:
    print("1. 결제 금액 컬럼을 찾을 수 없어 매출 검증 스킵")

# B. 행동 기반 검증
print(f"2. 실제 지점 방문 유저: {c7_checkin['client_uid'].nunique()}명")
print(f"3. '1일 지점 투어' 경험자(하루 2곳 이상): {tour_users['client_uid'].nunique()}명")

# C. 유입 상품 Top 5
print("\n4. 7번 그룹 유입 상품 Top 5:")
print(df_c7['product_name'].value_counts().head(5))

# D. 체크인 0회 유저 (노이즈 가능성)
ghost_users = set(cohort_7_uids) - set(c7_checkin['client_uid'].unique())
print(f"5. 결제 후 체크인 기록 전혀 없는 유저: {len(ghost_users)}명")

In [ ]:
df_check_in['sub_type'].unique()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import pandas as pd
import numpy as np
import warnings

# 1. 초기 설정
warnings.filterwarnings('ignore')
print("나눔고딕 폰트 적용 및 x축 정렬 중...")

# 폰트 설치 및 설정
!sudo apt-get install -qq fonts-nanum > /dev/null
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path)

# ---------------------------------------------------------
# 2. 데이터 가공 (행 기준 % 비중)
# ---------------------------------------------------------
try:
    # 코호트별 시간대 방문 비중 계산
    visit_pct_pivot = pd.crosstab(
        checkin_with_cohort['cohort'],
        checkin_with_cohort['hour'],
        normalize='index'
    ) * 100

    # 0~23시 전체 시간대 확보 (데이터 없는 시간도 0.0으로 표시)
    all_hours = list(range(24))
    visit_pct_pivot = visit_pct_pivot.reindex(columns=all_hours, fill_value=0)

except NameError:
    print("⚠️ 에러: 'checkin_with_cohort' 데이터가 없습니다.")

# ---------------------------------------------------------
# 3. 시각화 (x축 최적화 버전)
# ---------------------------------------------------------
if 'visit_pct_pivot' in locals():
    # 그래프 크기를 넉넉하게 설정하여 24개 숫자가 다 보이게 합니다.
    fig, ax = plt.subplots(figsize=(22, 10))

    # 히트맵 생성
    sns.heatmap(visit_pct_pivot,
                annot=True,
                fmt=".1f",
                cmap="YlGnBu",
                cbar_kws={'label': '방문 비중 (%)'},
                ax=ax)

    # [핵심] x축 눈금 설정: 모든 시간이 하나씩 다 나오게 강제 지정
    ax.set_xticks(np.arange(len(all_hours)) + 0.5) # 눈금을 칸 중앙에 배치
    ax.set_xticklabels(all_hours, fontsize=12)    # 0~23 숫자 표시

    # 타이틀 및 레이블 설정
    plt.title('코호트별 입실 시간대 분포 (%)', fontproperties=font_prop, fontsize=24, pad=30)
    plt.xlabel('입실 시간 ($0$시 - $23$시)', fontproperties=font_prop, fontsize=16)
    plt.ylabel('유저 코호트', fontproperties=font_prop, fontsize=16)

    # y축 코호트명 폰트 적용 및 수평 정렬
    ax.set_yticklabels(visit_pct_pivot.index, fontproperties=font_prop, fontsize=13, rotation=0)

    plt.tight_layout()
    plt.show()

    # 4. 요약 출력
    print("\n" + "="*70)
    print("■ 코호트별 최다 방문 시간대 (Peak Hour)")
    print("="*70)
    for cohort in visit_pct_pivot.index:
        peak_h = visit_pct_pivot.loc[cohort].idxmax()
        peak_v = visit_pct_pivot.loc[cohort].max()
        print(f"[{cohort:^25}] 피크타임: {peak_h:02d}시 (비중 {peak_v:.1f}%)")

In [ ]:
import pandas as pd

# 1. '순수 신규 진입자' 명단 추출 (6/11 ~ 12/31 사이 생애 첫 결제자)
analysis_start = '2025-06-11'
analysis_end = '2025-12-31'

# 원본 df에서 유저별 생애 첫 진입일을 다시 구합니다.
user_first_dates = df.groupby('client_uid')['start_date'].min().reset_index()
user_first_dates.rename(columns={'start_date': 'first_date_ever'}, inplace=True)

# 하반기에 처음 가입한 유저 리스트 (마케터님이 말씀하신 118명의 근거가 되는 모수)
pure_new_uids = user_first_dates[
    (user_first_dates['first_date_ever'] >= analysis_start) &
    (user_first_dates['first_date_ever'] <= analysis_end)
]['client_uid'].unique()

# 2. 3번 코호트 유저 중 위 리스트(순수 신규)에 포함된 유저 ID만 추출
target_cohort_name = '3. [진입] 기간권 첫 전환(Up-sell)'
uids_3_pure = user_summary[
    (user_summary['cohort'] == target_cohort_name) &
    (user_summary['client_uid'].isin(pure_new_uids))
]['client_uid'].unique()

print(f"■ 분석 대상: 3번 코호트 내 순수 신규 진입자 ({len(uids_3_pure)}명)")

# 3. 이들의 업셀링(2번째 결제) 상품 카테고리 추출
df_u3_purchases = df[df['client_uid'].isin(uids_3_pure)].sort_values(['client_uid', 'start_date'])
upsell_info = df_u3_purchases.groupby('client_uid').nth(1).reset_index()[['client_uid', '상품카테고리']]

# 4. 해당 118명의 체크인 행동 데이터 직접 집계
c3_pure_checkin = df_check_in[df_check_in['client_uid'].isin(uids_3_pure)].copy()
c3_pure_checkin['check_in_time'] = pd.to_datetime(c3_pure_checkin['check_in_time'])
c3_pure_checkin['hour'] = c3_pure_checkin['check_in_time'].dt.hour

user_behavior_c3_pure = c3_pure_checkin.groupby('client_uid').agg(
    총_방문횟수=('check_in_history_uid', 'count'),
    방문_지점수=('branch_uid', 'nunique'),
    주요_방문시간=('hour', lambda x: x.mode()[0] if not x.mode().empty else None)
).reset_index()

# 5. 데이터 병합 (상품 x 행동 x 계약정보)
final_pure_analysis = pd.merge(upsell_info, user_behavior_c3_pure, on='client_uid', how='inner')
final_pure_analysis = pd.merge(final_pure_analysis, user_summary[['client_uid', 'total_period_days']], on='client_uid', how='inner')

# 6. [최종 결과] 상품 카테고리별 요약
c3_pure_summary = final_pure_analysis.groupby('상품카테고리').agg(
    유저수=('client_uid', 'count'),
    평균_방문횟수=('총_방문횟수', 'mean'),
    평균_계약일=('total_period_days', 'mean'),
    주요_입실시간=('주요_방문시간', lambda x: x.mode()[0] if not x.mode().empty else None)
).reset_index()

# 방문 밀도(%) 계산
c3_pure_summary['방문_밀도(%)'] = (c3_pure_summary['평균_방문횟수'] / c3_pure_summary['평균_계약일'] * 100).round(1)
c3_pure_summary[['평균_방문횟수', '평균_계약일']] = c3_pure_summary[['평균_방문횟수', '평균_계약일']].round(1)

print(f"■ [순수 신규 {len(final_pure_analysis)}명 기준] 업셀링 상품군별 이용 패턴 분석")
display(c3_pure_summary.sort_values('유저수', ascending=False))

In [ ]:
import pandas as pd

# 1. 3번 코호트 유저 152명 리스트 확보
target_c = '3. [진입] 기간권 첫 전환(Up-sell)'
uids_3 = user_summary[user_summary['cohort'] == target_c]['client_uid'].unique()

# 2. 이들의 두 번째 상품(업셀링 상품) 카테고리 추출
upsell_info = df[df['client_uid'].isin(uids_3)].sort_values(['client_uid', 'start_date']).groupby('client_uid').nth(1).reset_index()
upsell_info = upsell_info[['client_uid', '상품카테고리']]

# 3. 이 152명의 체크인 시간대 데이터 확보
c3_checkins = df_check_in[df_check_in['client_uid'].isin(uids_3)].copy()
c3_checkins['hour'] = pd.to_datetime(c3_checkins['check_in_time']).dt.hour

# 4. 유저별 최빈값(가장 자주 오는 시간) 계산
user_mode_time = c3_checkins.groupby('client_uid')['hour'].agg(lambda x: x.mode()[0] if not x.mode().empty else None).reset_index()
user_mode_time.columns = ['client_uid', '주요_입실시간']

# 5. 상품군별로 모아서 결과 보기
# 3번 유저들의 [두번째 상품] + [자주 오는 시간] 결합
final_check = pd.merge(upsell_info, user_mode_time, on='client_uid')

# 결과 요약
mystery_solved = final_check.groupby(['상품카테고리', '주요_입실시간']).size().reset_index(name='유저수')
mystery_solved = mystery_solved.sort_values(['상품카테고리', '유저수'], ascending=[True, False])

print("■ [결과] 상품군별 유저들의 주요 입실 시간 분포")
display(mystery_solved)

In [ ]:
import pandas as pd

# 1. 지역 추출 함수 정의
def extract_region(addr):
    if not addr or pd.isna(addr):
        return "미분류"
    parts = addr.split()
    if len(parts) < 2:
        return "미분류"

    city = parts[0]    # 서울, 경기 등
    district = parts[1] # 강남구, 성남시 등

    # 서울은 '서울 구'까지, 나머지는 '도 시'까지 결합
    return f"{city} {district}"

# 2. 체크인 데이터와 코호트 정보 결합
# (df_check_in_filtered와 user_summary가 생성되어 있어야 합니다)
checkin_with_region = df_check_in_filtered.merge(
    user_summary[['client_uid', 'cohort']],
    on='client_uid',
    how='inner'
)

# 3. 주소에서 지역 정보 추출
checkin_with_region['region'] = checkin_with_region['street_address'].apply(extract_region)

# 4. 코호트별 지역 분포 (방문 횟수 비중 % 기준)
region_dist_pivot = pd.crosstab(
    checkin_with_region['cohort'],
    checkin_with_region['region'],
    normalize='index'
) * 100

# 5. 결과 출력 (소수점 1자리)
print("■ 코호트별 지역 이용 비중 (%)")
display(region_dist_pivot.round(1))

# 6. [인사이트] 코호트별 선호 지역 Top 3 추출
print("\n" + "="*50)
print("■ 코호트별 주요 활동 지역 (Top 3)")
print("="*50)
for cohort in region_dist_pivot.index:
    top_regions = region_dist_pivot.loc[cohort].nlargest(3)
    regions_str = ", ".join([f"{reg}({val:.1f}%)" for reg, val in top_regions.items()])
    print(f"[{cohort}] → {regions_str}")

In [ ]:
import pandas as pd

# 1. 코호트와 지역별로 그룹바이 하여 방문횟수 계산
# checkin_with_region 데이터프레임이 상단 코드에서 생성되어 있어야 합니다.
region_raw_df = checkin_with_region.groupby(['cohort', 'region']).size().reset_index(name='방문횟수')

# 2. 각 코호트별 총 방문횟수 계산 (비중 산출을 위한 분모)
cohort_total_visits = region_raw_df.groupby('cohort')['방문횟수'].transform('sum')

# 3. 방문비중(%) 계산
region_raw_df['방문비중(%)'] = (region_raw_df['방문횟수'] / cohort_total_visits * 100).round(1)

# 4. 가독성을 위해 코호트 순서와 비중이 높은 순으로 정렬
region_raw_df = region_raw_df.sort_values(by=['cohort', '방문비중(%)'], ascending=[True, False])

# 5. 결과 출력
print("■ [Raw Data] 코호트별 지역 이용 방문횟수 및 비중")
display(region_raw_df)

# 코호트별 인기지점

In [ ]:
import pandas as pd

# 1. 체크인 데이터와 코호트 정보 결합
# (df_check_in_filtered와 user_summary가 생성되어 있어야 합니다)
checkin_with_cohort = df_check_in_filtered.merge(
    user_summary[['client_uid', 'cohort']],
    on='client_uid',
    how='inner'
)

# 2. 코호트별 지점 방문 분포 계산 (방문 횟수 비중 % 기준)
# display_name 컬럼을 사용하여 지점별 비중을 산출합니다.
branch_dist_pivot = pd.crosstab(
    checkin_with_cohort['cohort'],
    checkin_with_cohort['display_name'],
    normalize='index'
) * 100

# 3. [인사이트] 코호트별 인기 지점 Top 3 추출
print("="*60)
print("■ [데이터] 코호트별 인기 지점 TOP 3 (방문 비중 %)")
print("="*60)

for cohort in branch_dist_pivot.index:
    # 해당 코호트에서 비중이 가장 높은 상위 3개 지점 추출
    top_branches = branch_dist_pivot.loc[cohort].nlargest(3)

    # 지점명과 비중을 문자열로 결합
    branches_str = ", ".join([f"{name}({val:.1f}%)" for name, val in top_branches.items()])

    print(f"[{cohort}]")
    print(f"  → {branches_str}")
    print("-" * 60)

In [ ]:
checkin_with_cohort.groupby('cohort')['client_uid'].nunique()

In [ ]:
# 모든 코호트 리스트 추출
all_cohorts = sorted(user_summary['cohort'].unique())

# 전체 코호트 대상 지점별 방문 횟수 및 비중 집계
checkin_with_cohort_all = df_check_in_filtered.merge(user_summary[['client_uid', 'cohort']], on='client_uid')
branch_usage_all = checkin_with_cohort_all.groupby(['cohort', 'display_name']).size().reset_index(name='방문횟수')
branch_usage_all['비중'] = branch_usage_all.groupby('cohort')['방문횟수'].transform(lambda x: (x / x.sum() * 100).round(1))

# 결과 출력 (코호트별 상위 10개 지점)
branch_rank_total = branch_usage_all.sort_values(['cohort', '방문횟수'], ascending=[True, False])

print("■ [전체 코호트] 지점 이용 상세 현황 (Top 10)")
display(branch_rank_total.groupby('cohort').head(10))

In [ ]:
import pandas as pd

# 1. 분석 기준일 설정 (코호트 종료일과 동일하게)
analysis_end = '2025-12-31'

# 2. 체크인 데이터 시간 변환 (이미 되어있다면 생략 가능)
df_check_in['check_in_time'] = pd.to_datetime(df_check_in['check_in_time'])

# 3. [핵심 수정] 유니버스 기간(6/11~12/31) 내의 체크인만 추출
df_check_in_cln = df_check_in[
    (df_check_in['check_in_time'] >= '2025-06-11') &
    (df_check_in['check_in_time'] <= analysis_end + ' 23:59:59')
].copy()

# 4. 6/11 이후 신규 진입자 리스트로 다시 필터링
new_joiners_list = new_joiners[new_joiners['start_date'] >= '2025-06-11']['client_uid']
df_check_in_final = df_check_in_cln[df_check_in_cln['client_uid'].isin(new_joiners_list)]

# 5. 유저별 이용 지점 수 재계산
user_branch_cnt_fixed = df_check_in_final.groupby('client_uid')['display_name'].nunique().reset_index(name='이용지점수')

# 6. 코호트 정보와 결합 및 피벗 테이블 생성
branch_analysis_fixed = user_branch_cnt_fixed.merge(user_summary[['client_uid', 'cohort']], on='client_uid')

def group_branch_cnt(cnt):
    if cnt <= 5: return f"{cnt}개"
    elif cnt <= 10: return "6-10개"
    else: return "11개+"

branch_analysis_fixed['지점수_그룹'] = branch_analysis_fixed['이용지점수'].apply(group_branch_cnt)

# 7. 최종 사람 수(Count) 확인
final_count_pivot_fixed = pd.crosstab(
    branch_analysis_fixed['지점수_그룹'],
    branch_analysis_fixed['cohort']
)

# 정렬
cohort_order = sorted(branch_analysis_fixed['cohort'].unique())
final_count_pivot_fixed = final_count_pivot_fixed.reindex(
    index=['1개', '2개', '3개', '4개', '5개', '6-10개', '11개+'],
    columns=cohort_order
).fillna(0).astype(int)

print(f"■ [수정 완료] 12/31까지의 기록만 반영한 지점수별 유저 수")
display(final_count_pivot_fixed)

In [ ]:
import pandas as pd

# 1. 유저별 이용 지점 수 계산
user_branch_cnt = df_check_in_filtered.groupby('client_uid')['branch_uid'].nunique().reset_index(name='이용지점수')

# 2. 유저 정보 결합 (코호트 정보 추가)
branch_analysis = user_branch_cnt.merge(user_summary[['client_uid', 'cohort']], on='client_uid')

# 3. [핵심] 지점 수별 유저 수 피벗 테이블 생성 (값에 count 적용)
cohort_count_by_cnt = pd.crosstab(
    branch_analysis['이용지점수'],
    branch_analysis['cohort']
)

# 4. 가독성을 위해 지점수 그룹화 (필요시)
def group_branch_cnt(cnt):
    if cnt <= 5: return f"{cnt}개"
    elif cnt <= 10: return "6-10개"
    else: return "11개+"

branch_analysis['지점수_그룹'] = branch_analysis['이용지점수'].apply(group_branch_cnt)

# 그룹화된 기준으로 사람 수 집계
final_count_pivot = pd.crosstab(
    branch_analysis['지점수_그룹'],
    branch_analysis['cohort']
)

# 코호트 순서 및 지점수 그룹 순서 정렬
cohort_order = sorted(branch_analysis['cohort'].unique())
final_count_pivot = final_count_pivot.reindex(
    index=['1개', '2개', '3개', '4개', '5개', '6-10개', '11개+'],
    columns=cohort_order
).fillna(0).astype(int)

print("■ [검증] 이용 지점 수에 따른 코호트별 실제 유저 수 (명)")
display(final_count_pivot)

In [ ]:
import pandas as pd

# 1. 유저별 이용 지점 수 계산
user_branch_cnt = df_check_in_filtered.groupby('client_uid')['branch_uid'].nunique().reset_index(name='이용지점수')

# 2. 유저 정보 결합
branch_analysis = user_branch_cnt.merge(user_summary[['client_uid', 'cohort']], on='client_uid')

# 3. 지점수 그룹화 함수
def group_branch_cnt(cnt):
    if cnt <= 5: return f"{cnt}개"
    elif cnt <= 10: return "6-10개"
    else: return "11개+"

branch_analysis['지점수_그룹'] = branch_analysis['이용지점수'].apply(group_branch_cnt)

# 4. [수정 포인트] 비중(%)으로 피벗 테이블 생성 (normalize='index' 적용)
final_ratio_pivot = pd.crosstab(
    branch_analysis['지점수_그룹'],
    branch_analysis['cohort'],
    normalize='index' # 행(지점수 그룹) 기준으로 합계를 1(100%)로 만듦
) * 100

# 5. 정렬 및 포맷팅
cohort_order = sorted(branch_analysis['cohort'].unique())
final_ratio_pivot = final_ratio_pivot.reindex(
    index=['1개', '2개', '3개', '4개', '5개', '6-10개', '11개+'],
    columns=cohort_order
).fillna(0)

print("■ [검증] 이용 지점 수 그룹별 코호트 구성 비중 (%)")
display(final_ratio_pivot.round(1)) # 소수점 첫째자리까지 표시

In [ ]:
df_check_in_filtered.head()

In [ ]:
# 1. 12/31까지의 체크인 데이터 필터링 (다시 한번 확실히)
df_check_in['check_in_time'] = pd.to_datetime(df_check_in['check_in_time'])
df_25_checkin = df_check_in[df_check_in['check_in_time'] <= '2025-12-31 23:59:59'].copy()

# 2. 7번 코호트 유저 중 지점수가 5개 이상인 유저만 추출
c7_uids = user_summary[user_summary['cohort'] == '7. [신규] 단순 체험']['client_uid']
c7_heavy_checkin = df_25_checkin[df_25_checkin['client_uid'].isin(c7_uids)]

# 3. 유저별 지점 수 다시 계산
c7_branch_counts = c7_heavy_checkin.groupby('client_uid')['display_name'].nunique().reset_index(name='real_branch_cnt')
target_uids = c7_branch_counts[c7_branch_counts['real_branch_cnt'] >= 5]['client_uid']

# 4. [핵심] 이들의 상세 로그 확인 (3613 포함)
# 이 유저들이 '어떤 날' '어떤 지점'을 '몇 시'에 찍었는지 봅니다.
detail_log = c7_heavy_checkin[c7_heavy_checkin['client_uid'].isin(target_uids)].sort_values(['client_uid', 'check_in_time'])

print("■ 7번 코호트 중 다지점 이용 유저의 상세 체크인 로그 (상위 30개)")
display(detail_log[['client_uid', 'check_in_time', 'display_name', 'sub_type']].head(30))

# 5. [가설 검증] 하루에 여러 지점을 찍었는가?
daily_visit = detail_log.groupby(['client_uid', 'check_in_date'])['display_name'].nunique().reset_index(name='daily_branch_cnt')
print("\n■ 하루 최대 방문 지점 수")
display(daily_visit.sort_values('daily_branch_cnt', ascending=False).head(10))

In [ ]:
# 1. 12/31까지의 기록을 기준으로 7번 코호트 + 11개 이상 이용자 추출
target_uids = branch_analysis_fixed[
    (branch_analysis_fixed['cohort'] == '7. [신규] 단순 체험') &
    (branch_analysis_fixed['이용지점수'] >= 11)
]['client_uid'].tolist()

# 2. 이들의 요약 정보 조회 (LTV, 구매 상품 등)
c7_heavy_users = user_summary[user_summary['client_uid'].isin(target_uids)].copy()

# 3. 가독성을 위해 필요한 컬럼만 추출하여 테이블 출력
print(f"■ 7번 코호트(신규 단순 체험) 중 11개 이상 지점 이용자 명단 (총 {len(target_uids)}명)")

# 유저별 상세 정보 (첫 구매 상품 및 누적 이용일 포함)
display_cols = ['client_uid', 'cohort', 'total_buy_count', 'total_revenue', 'total_period_days', 'first_product_cat']
display(c7_heavy_users[display_cols].sort_values('total_period_days', ascending=False))

# 4. (참고) 이들이 실제로 어떤 지점들을 돌아다녔는지 로그 확인 (상위 5명 샘플)
print("\n■ 상기 유저들의 실제 체크인 지점 로그 (샘플)")
sample_logs = df_check_in_final[df_check_in_final['client_uid'].isin(target_uids[:5])]
display(sample_logs.groupby('client_uid')['display_name'].apply(lambda x: list(x.unique())).reset_index())

In [ ]:
# 3613 유저의 상세 이용 기록 확인
user_3613 = df[df['client_uid'] == 5162]
user_3613

In [ ]:
import pandas as pd

# 1. 체크인 데이터와 계약 데이터(df_clean)를 결합
# df_clean에 포함된 '특정 계약 기간' 내의 기록만 보기 위함입니다.
df_check_in_strict = df_check_in.merge(
    df_clean[['client_uid', 'start_date', 'actual_end_date']],
    on='client_uid',
    how='inner'
)

# 2. [핵심 필터] 체크인 시간이 계약 시작일과 종료일(익일 새벽까지) 사이인 것만 필터링
df_check_in_strict['check_in_time'] = pd.to_datetime(df_check_in_strict['check_in_time'])
df_check_in_strict = df_check_in_strict[
    (df_check_in_strict['check_in_time'] >= df_check_in_strict['start_date']) &
    (df_check_in_strict['check_in_time'] <= df_check_in_strict['actual_end_date'] + pd.Timedelta(days=1))
].copy()

# 3. 유저별 이용 지점 수 재계산
user_branch_strict = df_check_in_strict.groupby('client_uid')['display_name'].nunique().reset_index(name='이용지점수')

# 4. 코호트 정보 결합 및 가독성 패치 (+11 표기)
branch_analysis_final = user_branch_strict.merge(user_summary[['client_uid', 'cohort']], on='client_uid')

def group_label_updated(cnt):
    if cnt <= 5: return str(cnt)
    elif cnt <= 10: return "6-10"
    else: return "+11"  # 가독성을 위해 +11로 변경

branch_analysis_final['지점수_그룹'] = branch_analysis_final['이용지점수'].apply(group_label_updated)

# 5. [결과 1] 지점수 그룹별 코호트 구성 (인원 수)
final_pivot_cnt = pd.crosstab(branch_analysis_final['지점수_그룹'], branch_analysis_final['cohort'])
order = ['1', '2', '3', '4', '5', '6-10', '+11']
final_pivot_cnt = final_pivot_cnt.reindex(order).fillna(0).astype(int)

# 6. [결과 2] 7번 코호트 중 여전히 +11인 '진짜 헤비 체험자' 리스트 추출
c7_heavy_list = branch_analysis_final[
    (branch_analysis_final['cohort'].str.contains('7.')) &
    (branch_analysis_final['이용지점수'] >= 11)
].merge(user_summary[['client_uid', 'first_product_cat', 'total_revenue']], on='client_uid')

print("■ [검증 완료] 계약 기간 내 활동만 반영한 지점수별 코호트 현황")
display(final_pivot_cnt)

print(f"\n■ [상세조회] 7번 코호트 중 진짜로 11개 이상 투어한 유저 (총 {len(c7_heavy_list)}명)")
display(c7_heavy_list)

In [ ]:
import pandas as pd

# 1. 지점수 그룹별 코호트 비중(%) 피벗 테이블 생성
# normalize='columns'를 사용하여 각 코호트(열)의 합을 100%로 만듭니다.
final_cohort_ratio_pivot = pd.crosstab(
    branch_analysis_final['지점수_그룹'],
    branch_analysis_final['cohort'],
    normalize='columns' # <-- 열(코호트) 기준으로 100% 계산
) * 100

# 2. 정렬 (1, 2, 3, 4, 5, 6-10, +11)
order = ['1', '2', '3', '4', '5', '6-10', '+11']
cohort_order = sorted(branch_analysis_final['cohort'].unique())
final_cohort_ratio_pivot = final_cohort_ratio_pivot.reindex(index=order, columns=cohort_order).fillna(0)

print("■ [전략 분석] 각 코호트 내 지점 이용수 분포 (%)")
print("- '각 열의 합계 = 100%' 이며, 해당 그룹 유저들의 행동 패턴을 보여줍니다.")
display(final_cohort_ratio_pivot.round(1))

# 결과 확인

In [ ]:
# df_NaN = region_raw_df.astype(str) (기존 코드 유지)

# gspread 인증 및 스프레드시트 열기
# 깃허브 업로드용 마스킹 버전으로 경로와 ID 변경
gc = gspread.service_account(filename="YOUR_SERVICE_ACCOUNT_KEY.json")

sheet_id = "YOUR_GOOGLE_SPREADSHEET_ID"
worksheet_name = "region_raw_df"
sh = gc.open_by_key(sheet_id)
worksheet = sh.worksheet(worksheet_name)

# (이후 하단 데이터 업로드 로직은 그대로 유지)